# Superstore SQL Analysis

Independent analysis for Assignment 3 only. The goal here is to load the Superstore CSV into SQLite, split it into a few cleaner tables, and then answer the business questions with subqueries, CTEs, and window functions.

In [3]:
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

base_path = Path('c:/Users/Subham Singhania/Desktop/celebal/Assignment-3')
csv_path = base_path / 'Sample - Superstore.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1') 
df.columns = [
    col.strip().lower().replace(' ', '_').replace('-', '_')
    for col in df.columns
]

for col in ['order_date', 'ship_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')

display(df.head())
print(f'rows: {len(df):,}')
print(f'columns: {len(df.columns):,}')

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


rows: 9,994
columns: 21


In [4]:
conn = sqlite3.connect(':memory:')
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

conn.executescript('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT DISTINCT
    customer_id,
    customer_name,
    segment,
    country,
    city,
    state,
    postal_code,
    region
FROM superstore_raw;

DROP TABLE IF EXISTS orders;
CREATE TABLE orders AS
SELECT DISTINCT
    order_id,
    order_date,
    ship_date,
    ship_mode,
    customer_id,
    segment,
    country,
    city,
    state,
    postal_code,
    region
FROM superstore_raw;

DROP TABLE IF EXISTS products;
CREATE TABLE products AS
SELECT DISTINCT
    product_id,
    category,
    sub_category,
    product_name
FROM superstore_raw;
''')

table_counts = pd.read_sql_query(
    '''
    SELECT 'superstore_raw' AS table_name, COUNT(*) AS row_count FROM superstore_raw
    UNION ALL
    SELECT 'customers', COUNT(*) FROM customers
    UNION ALL
    SELECT 'orders', COUNT(*) FROM orders
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    ''',
    conn,
)

display(table_counts)

,table_name,row_count
0,superstore_raw,9994
1,customers,4910
2,orders,5009
3,products,1894


In [5]:
def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

above_average_sales = q('''
    SELECT
        order_id,
        customer_id,
        customer_name,
        product_id,
        product_name,
        sales,
        profit
    FROM superstore_raw
    WHERE sales > (SELECT AVG(sales) FROM superstore_raw)
    ORDER BY sales DESC
    LIMIT 10
''')

display(above_average_sales)

highest_order_per_customer = q('''
    WITH order_totals AS (
        SELECT
            o.customer_id,
            c.customer_name,
            o.order_id,
            SUM(r.sales) AS order_sales
        FROM superstore_raw r
        JOIN orders o ON r.order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY o.customer_id, c.customer_name, o.order_id
    )
    SELECT
        customer_id,
        customer_name,
        order_id,
        order_sales
    FROM order_totals ot
    WHERE order_sales = (
        SELECT MAX(order_sales)
        FROM order_totals
        WHERE customer_id = ot.customer_id
    )
    ORDER BY order_sales DESC
    LIMIT 10
''')

display(highest_order_per_customer)

,order_id,customer_id,customer_name,product_id,product_name,sales,profit
0,CA-2014-145317,SM-20320,Sean Miller,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784
1,CA-2016-118689,TC-20980,Tamara Chand,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,17499.950,8399.9760
2,CA-2017-140151,RB-19360,Raymond Buch,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808
3,CA-2017-127180,TA-21385,Tom Ashbrook,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,11199.968,3919.9888
4,CA-2017-166709,HL-15040,Hunter Lopez,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,10499.970,5039.9856
5,CA-2016-117121,AB-10105,Adrian Barton,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,9892.740,4946.3700
6,CA-2014-116904,SC-20095,Sanjit Chand,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,9449.950,4630.4755
7,US-2016-107440,BS-11365,Bill Shonely,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,2365.9818
8,CA-2016-158841,SE-20110,Sanjit Engle,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,8749.950,2799.9840
9,CA-2016-143714,CC-12370,Christopher Conant,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,8399.976,1119.9968


,customer_id,customer_name,order_id,order_sales
0,SM-20320,Sean Miller,CA-2014-145317,118306.14
1,SE-20110,Sanjit Engle,CA-2016-158841,96855.44
2,TC-20980,Tamara Chand,CA-2016-118689,91683.70
3,KL-16645,Ken Lonsdale,CA-2014-143917,91512.19
4,SC-20095,Sanjit Chand,CA-2014-116904,89101.71
5,AB-10105,Adrian Barton,CA-2016-117121,89034.66
6,RB-19360,Raymond Buch,CA-2017-140151,84314.88
7,SV-20365,Seth Vernon,CA-2017-100111,73599.18
8,HL-15040,Hunter Lopez,CA-2017-166709,62999.82
9,GT-14710,Greg Tran,CA-2014-128209,61606.16


In [6]:
customer_rankings = q('''
    WITH customer_sales AS (
        SELECT
            c.customer_id,
            c.customer_name,
            c.segment,
            SUM(r.sales) AS total_sales,
            COUNT(DISTINCT o.order_id) AS total_orders
        FROM superstore_raw r
        JOIN customers c ON r.customer_id = c.customer_id
        JOIN orders o ON r.order_id = o.order_id
        GROUP BY c.customer_id, c.customer_name, c.segment
    ),
    ranked_customers AS (
        SELECT
            customer_id,
            customer_name,
            segment,
            total_sales,
            total_orders,
            ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_number_rank,
            RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
        FROM customer_sales
    )
    SELECT *
    FROM ranked_customers
    ORDER BY sales_rank, customer_name
''')

display(customer_rankings.head(10))
display(customer_rankings.tail(10))

single_order_customers = customer_rankings[customer_rankings['total_orders'] == 1].copy()
display(single_order_customers.sort_values('total_sales', ascending=False).head(10))

,customer_id,customer_name,segment,total_sales,total_orders,row_number_rank,sales_rank
0,KL-16645,Ken Lonsdale,Consumer,155927.519,12,1,1
1,SE-20110,Sanjit Engle,Consumer,134303.818,11,2,2
2,CL-12565,Clay Ludtke,Consumer,130566.552,12,3,3
3,AB-10105,Adrian Barton,Consumer,130262.139,10,4,4
4,SC-20095,Sanjit Chand,Consumer,127281.006,9,5,5
5,SM-20320,Sean Miller,Home Office,125215.250,5,6,6
6,EH-13765,Edward Hooks,Corporate,123730.560,12,7,7
7,GT-14710,Greg Tran,Consumer,118201.200,11,8,8
8,SV-20365,Seth Vernon,Consumer,114709.500,10,9,9
9,JL-15835,John Lee,Consumer,107799.153,11,10,10


,customer_id,customer_name,segment,total_sales,total_orders,row_number_rank,sales_rank
783,SG-20890,Susan Gilcrest,Corporate,143.838,3,784,784
784,AS-10135,Adrian Shami,Home Office,117.640,2,785,785
785,RM-19750,Roland Murray,Consumer,98.350,1,786,786
786,AR-10570,Anemone Ratner,Consumer,88.150,1,787,787
787,RE-19405,Ricardo Emerson,Consumer,48.360,1,788,788
788,RS-19870,Roy Skaria,Home Office,44.656,2,789,789
789,MG-18205,Mitch Gastineau,Corporate,16.739,1,790,790
790,CJ-11875,Carl Jackson,Corporate,16.520,1,791,791
791,TS-21085,Thais Sissman,Consumer,9.666,2,792,792
792,LD-16855,Lela Donovan,Corporate,5.304,1,793,793


,customer_id,customer_name,segment,total_sales,total_orders,row_number_rank,sales_rank
747,JC-15385,Jenna Caffey,Consumer,1058.108,1,748,748
748,SM-20905,Susan MacKendrick,Consumer,1043.041,1,749,749
749,TC-21145,Theresa Coyne,Corporate,1038.260,1,750,750
756,JR-15700,Jocasta Rupert,Consumer,863.880,1,757,757
761,PH-18790,Patricia Hirasaki,Home Office,729.648,1,762,762
781,AO-10810,Anthony O'Donnell,Corporate,161.280,1,782,782
785,RM-19750,Roland Murray,Consumer,98.350,1,786,786
786,AR-10570,Anemone Ratner,Consumer,88.150,1,787,787
787,RE-19405,Ricardo Emerson,Consumer,48.360,1,788,788
789,MG-18205,Mitch Gastineau,Corporate,16.739,1,790,790


In [7]:
top_customers = customer_rankings.head(10).copy()
low_customers = customer_rankings.sort_values(['total_sales', 'customer_name']).head(10).copy()
above_average_count = len(above_average_sales)
single_order_count = len(single_order_customers)

highest_customer = customer_rankings.iloc[0]
lowest_customer = customer_rankings.iloc[-1]

print('Top customers by total sales:')
print(top_customers[['customer_name', 'total_sales', 'total_orders']].to_string(index=False))
print()
print('Low customers by total sales:')
print(low_customers[['customer_name', 'total_sales', 'total_orders']].to_string(index=False))
print()
print('Business notes:')
print(f'- Line items above average sales in the first sample slice: {above_average_count}')
print(f'- Single-order customers found in the dataset: {single_order_count}')
print(f'- Highest ranked customer: {highest_customer["customer_name"]} with sales {highest_customer["total_sales"]:.2f}')
print(f'- Bottom customer in the ranking slice: {lowest_customer["customer_name"]} with sales {lowest_customer["total_sales"]:.2f}')

Top customers by total sales:
customer_name  total_sales  total_orders
 Ken Lonsdale   155927.519            12
 Sanjit Engle   134303.818            11
  Clay Ludtke   130566.552            12
Adrian Barton   130262.139            10
 Sanjit Chand   127281.006             9
  Sean Miller   125215.250             5
 Edward Hooks   123730.560            12
    Greg Tran   118201.200            11
  Seth Vernon   114709.500            10
     John Lee   107799.153            11

Low customers by total sales:
  customer_name  total_sales  total_orders
   Lela Donovan        5.304             1
  Thais Sissman        9.666             2
   Carl Jackson       16.520             1
Mitch Gastineau       16.739             1
     Roy Skaria       44.656             2
Ricardo Emerson       48.360             1
 Anemone Ratner       88.150             1
  Roland Murray       98.350             1
   Adrian Shami      117.640             2
 Susan Gilcrest      143.838             3

Business notes

## Quick read

The interesting part is not just who sells the most, but how concentrated the sales are. A few customers usually carry a very large chunk of revenue, while a long tail of single-order customers sits near the bottom. That is the part I would actually act on first: keep the heavy buyers warm, and try to convert the one-time buyers into repeat customers.